<a href="https://colab.research.google.com/github/Pedro4Albuquerque/Processamento-de-Linguagem-Natural/blob/main/Aula_9_Pratica_Passo_a_Passo_parte2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# Instalação e download de recursos (rodar 1x)
!pip install -q spacy scikit-learn pandas
!python -m spacy download pt_core_news_sm -q

import random
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline

nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado com sucesso.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 78.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Ambiente configurado com sucesso.


In [12]:
aspectos = ["O atendimento", "O corretor", "A visita", "A negociação", "O apartamento",
            "O condomínio", "A entrega das chaves", "O contrato", "O financiamento", "A vistoria"]

avaliacoes_positivas = ["foi excelente", "foi rápido e eficiente", "superou minhas expectativas",
                        "foi ótimo, recomendo", "foi tranquilo do início ao fim",
                        "me deixou muito satisfeito", "foi muito bem feito",
                        "foi claro e organizado", "foi perfeito", "merece nota dez"]

avaliacoes_negativas = ["foi péssimo", "atrasou demais", "me deixou decepcionado",
                        "foi ruim, não recomendo", "foi confuso e demorado", "não foi resolvido",
                        "deu muito problema", "foi desorganizado", "foi uma decepção",
                        "não atendeu ao combinado"]

random.seed(42)
positivos = random.sample([f"{a} {p}." for a in aspectos for p in avaliacoes_positivas], 50)
negativos = random.sample([f"{a} {n}." for a in aspectos for n in avaliacoes_negativas], 50)

textos = positivos + negativos
rotulos = ["Positivo"] * 50 + ["Negativo"] * 50

print(f"Total de avaliações: {len(textos)}")
for t, r in list(zip(textos, rotulos))[:3] + list(zip(textos, rotulos))[50:53]:
    print(f"{r:9} | {t}")

Total de avaliações: 100
Positivo  | O financiamento foi rápido e eficiente.
Positivo  | O corretor foi tranquilo do início ao fim.
Positivo  | O atendimento foi ótimo, recomendo.
Negativo  | A entrega das chaves foi uma decepção.
Negativo  | O corretor não foi resolvido.
Negativo  | O apartamento foi uma decepção.


In [13]:
PALAVRAS_NEGACAO = {"não", "nunca", "jamais", "nada", "ninguém"}

def lematizar(texto):
    doc = nlp(texto)
    return " ".join(
        token.lemma_.lower() for token in doc
        if not token.is_punct and (not token.is_stop or token.lower_ in PALAVRAS_NEGACAO)
    )

corpus_lematizado = [lematizar(t) for t in textos]
print(corpus_lematizado[:3])
print(corpus_lematizado[50:53])

['financiamento rápido eficiente', 'corretor tranquilo início', 'atendimento bom recomendar']
['entrega chave decepção', 'corretor não resolver', 'apartamento decepção']


In [14]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    corpus_lematizado, rotulos,
    test_size=0.25,
    random_state=42,
    stratify=rotulos
)

print(f"Treino: {len(X_treino)} avaliações | Teste: {len(X_teste)} avaliações")

from collections import Counter
print(Counter(y_treino), Counter(y_teste))

Treino: 75 avaliações | Teste: 25 avaliações
Counter({'Positivo': 38, 'Negativo': 37}) Counter({'Negativo': 13, 'Positivo': 12})


In [15]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    corpus_lematizado, rotulos,
    test_size=0.25,
    random_state=42,
    stratify=rotulos
)

print(f"Treino: {len(X_treino)} avaliações | Teste: {len(X_teste)} avaliações")

from collections import Counter
print(Counter(y_treino), Counter(y_teste))

Treino: 75 avaliações | Teste: 25 avaliações
Counter({'Positivo': 38, 'Negativo': 37}) Counter({'Negativo': 13, 'Positivo': 12})


In [16]:
vectorizer = TfidfVectorizer()

X_treino_vet = vectorizer.fit_transform(X_treino)

X_teste_vet = vectorizer.transform(X_teste)

print("Formato treino:", X_treino_vet.shape)

print("Formato teste: ", X_teste_vet.shape)


Formato treino: (75, 42)
Formato teste:  (25, 42)


In [17]:
modelo_nb = MultinomialNB()
modelo_nb.fit(X_treino_vet, y_treino)

modelo_lr = LogisticRegression(max_iter=1000)
modelo_lr.fit(X_treino_vet, y_treino)

print("Modelos treinados.")

Modelos treinados.


In [26]:
for nome, modelo in [("Naive Bayes", modelo_nb), ("Regressão Logística", modelo_lr)]:
    previsoes = modelo.predict(X_teste_vet)
    print(f"=== {nome} ===")
    print("Acurácia:", accuracy_score(y_teste, previsoes))
    print("Matriz de confusão (linhas = real, colunas = previsto):")
    print(confusion_matrix(y_teste, previsoes, labels=["Negativo", "Positivo"]))
    print(classification_report(y_teste, previsoes, digits=2))

=== Naive Bayes ===
Acurácia: 1.0
Matriz de confusão (linhas = real, colunas = previsto):
[[13  0]
 [ 0 12]]
              precision    recall  f1-score   support

    Negativo       1.00      1.00      1.00        13
    Positivo       1.00      1.00      1.00        12

    accuracy                           1.00        25
   macro avg       1.00      1.00      1.00        25
weighted avg       1.00      1.00      1.00        25

=== Regressão Logística ===
Acurácia: 1.0
Matriz de confusão (linhas = real, colunas = previsto):
[[13  0]
 [ 0 12]]
              precision    recall  f1-score   support

    Negativo       1.00      1.00      1.00        13
    Positivo       1.00      1.00      1.00        12

    accuracy                           1.00        25
   macro avg       1.00      1.00      1.00        25
weighted avg       1.00      1.00      1.00        25



In [27]:
for nome, modelo in [("Naive Bayes", MultinomialNB()), ("Regressão Logística", LogisticRegression(max_iter=1000))]:

    pipeline = make_pipeline(TfidfVectorizer(), modelo)

    notas = cross_val_score(pipeline, corpus_lematizado, rotulos, cv=5)

    print(f"{nome}: {notas.round(2)} -> média {notas.mean():.2f}")


Naive Bayes: [1. 1. 1. 1. 1.] -> média 1.00
Regressão Logística: [1. 1. 1. 1. 1.] -> média 1.00


In [28]:
print("Classes:", modelo_lr.classes_)

pesos = modelo_lr.coef_[0]
palavras = vectorizer.get_feature_names_out()

tabela_pesos = pd.DataFrame({"Palavra": palavras, "Peso": pesos.round(2)}).sort_values("Peso")
print("Mais negativas:")
print(tabela_pesos.head(6).to_string(index=False))
print("\nMais positivas:")
print(tabela_pesos.tail(6).iloc[::-1].to_string(index=False))

Classes: ['Negativo' 'Positivo']
Mais negativas:
     Palavra  Peso
         não -1.35
    decepção -1.12
     atrasar -0.97
desorganizar -0.96
     péssimo -0.83
    resolver -0.78

Mais positivas:
   Palavra  Peso
       bom  0.97
     fazer  0.97
satisfeito  0.91
 excelente  0.77
   merecer  0.77
      nota  0.77


In [29]:
frases_novas = [
    "O corretor foi excelente e muito atencioso.",
    "O condomínio não foi bom, deu problema.",
    "Nossa, que ótimo, mais um atraso na entrega.",
    "Visita agendada rapidinho, equipe nota dez.",
]

vetores_novos = vectorizer.transform([lematizar(f) for f in frases_novas])
previsoes = modelo_lr.predict(vetores_novos)
probabilidades = modelo_lr.predict_proba(vetores_novos)

for frase, previsao, prob in zip(frases_novas, previsoes, probabilidades):
    print(f"{previsao:9} (Neg={prob[0]:.2f} | Pos={prob[1]:.2f}) | {frase}")

Positivo  (Neg=0.37 | Pos=0.63) | O corretor foi excelente e muito atencioso.
Negativo  (Neg=0.82 | Pos=0.18) | O condomínio não foi bom, deu problema.
Positivo  (Neg=0.45 | Pos=0.55) | Nossa, que ótimo, mais um atraso na entrega.
Positivo  (Neg=0.45 | Pos=0.55) | Visita agendada rapidinho, equipe nota dez.


In [30]:
vocabulario = set(vectorizer.get_feature_names_out())

for frase in frases_novas:
    lematizada = lematizar(frase)
    reconhecidas = [p for p in lematizada.split() if p in vocabulario]
    print(frase)
    print(f"   lematizada:   {lematizada}")
    print(f"   reconhecidas: {reconhecidas}")
    print()

O corretor foi excelente e muito atencioso.
   lematizada:   corretor excelente atencioso
   reconhecidas: ['corretor', 'excelente']

O condomínio não foi bom, deu problema.
   lematizada:   condomínio não dar problema
   reconhecidas: ['condomínio', 'não', 'dar', 'problema']

Nossa, que ótimo, mais um atraso na entrega.
   lematizada:   ótimo atraso entrega
   reconhecidas: ['entrega']

Visita agendada rapidinho, equipe nota dez.
   lematizada:   visita agendar rapidinho equipe noto
   reconhecidas: ['visita']

